# Lesson 6.1 — The Failure Taxonomy: Integration Events

Failures are named integration events, each tied to a signal the layers already emit, split into hard failures and health warnings.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, run_pipeline, localize, failure_event,
    FAILURE_TAXONOMY, REACH_MAX)
W = GreenhouseWorld.demo_row(n=6, seed=1)
ALL_IDS = [f.fid for f in W.fruit]
TGT = understand(model_perception(W, rng=np.random.default_rng(7)), W)[1]
def big(t, j): return 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []
expected = {'NO_TARGET','UNREACHABLE','PLAN_INVALID','TRACKING_FAILURE',
            'NEAR_SINGULAR','EXCESS_EFFORT'}
print('taxonomy events:', sorted(FAILURE_TAXONOMY))
checks.append(set(FAILURE_TAXONOMY) == expected)
for code, e in FAILURE_TAXONOMY.items():
    checks.append(all(k in e for k in ('what','where','who','signal')))


taxonomy events: ['EXCESS_EFFORT', 'NEAR_SINGULAR', 'NO_TARGET', 'PLAN_INVALID', 'TRACKING_FAILURE', 'UNREACHABLE']


### failure_event builds an event with a severity

In [2]:
ev = failure_event('PLAN_INVALID')
warn = failure_event('NEAR_SINGULAR', severity='warning', detail=0.01)
print('PLAN_INVALID severity:', ev['severity'], '| NEAR_SINGULAR severity:', warn['severity'])
checks.append(ev['severity'] == 'failure' and warn['severity'] == 'warning')


PLAN_INVALID severity: failure | NEAR_SINGULAR severity: warning


### The taxonomy on real runs: occlusion -> NO_TARGET, disturbance -> TRACKING_FAILURE

In [3]:
r_occ = run_pipeline(W, W.q.copy(), perception=dict(occlude=ALL_IDS))
r_dis = run_pipeline(W, W.q.copy(), disturbance=big)
print('occlude-all events:', [e['code'] for e in r_occ['events']])
print('disturbance events:', [(e['code'], e['severity']) for e in r_dis['events']])
checks.append(any(e['code']=='NO_TARGET' for e in r_occ['events']))
checks.append(any(e['code']=='TRACKING_FAILURE' for e in r_dis['events']))
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


occlude-all events: ['NO_TARGET']
disturbance events: [('TRACKING_FAILURE', 'failure'), ('EXCESS_EFFORT', 'warning')]
10/10 checks passed.
All checks passed.
